# Install Dependencies

In [ ]:
%pip install torch --index-url https://download.pytorch.org/whl/cu128
%pip install "trl>=0.20.0" "peft>=0.17.0" "transformers>=4.55.0"
!pip install -U bitsandbytes

Import

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel


In [ ]:
MODEL_PATH = "smebellis/apt_threats"

# Load Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=True)

# Load Base Model

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Instruct-2507",  # same base model used for training
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
# Resize token embeddings to match the tokenizer's vocabulary size
base_model.resize_token_embeddings(len(tokenizer))

In [ ]:
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model.eval()

# Helper Function to Generate Prediction

In [ ]:
def generate_iocs(prompt_text, max_new_tokens=256):
    messages = [
        {"role": "system", "content": "You are a cyber threat intelligence model trained to extract IOCs."},
        {"role": "user", "content": prompt_text}
    ]
    # First, get the formatted chat string. tokenize=False ensures it returns a string.
    formatted_chat = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )
    # Then, tokenize the formatted string to get a BatchEncoding object (dictionary of tensors).
    enc = tokenizer(
        formatted_chat,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}

    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            repetition_penalty=1.05
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# =======Example Usuage==========

In [ ]:
example_text = """
Generate IOCs for APT1
"""



In [ ]:
print("=== Inference Output ===")
print(generate_iocs(example_text))